# Task 5: Bronze Ingestion

In [0]:
from pyspark.sql.functions import input_file_name, current_date, current_timestamp, col

In [0]:
CATALOG_NAME = 'dbr_dev_ua5816bd'
STORAGE_ACCOUNT = 'dlsua5816bd'
LOGIN = 'oles0305'

adls_url = f'abfss://{LOGIN}@{STORAGE_ACCOUNT}.dfs.core.windows.net'

RAW_PATH = adls_url + '/raw_data/'
CHECKPOINT_PATH = adls_url + '/bronze/_checkpoints/drivers'
TARGET_TABLE = f'{CATALOG_NAME}.{LOGIN}_bronze.drivers'

####Explicitly define the schema

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType

drivers_schema = StructType([
    StructField("driverId", IntegerType(), False),
    StructField("driverRef", StringType(), True),
    StructField("number", IntegerType(), True), 
    StructField("code", StringType(), True),
    StructField("forename", StringType(), True),
    StructField("surname", StringType(), True),
    StructField("dob", DateType(), True), 
    StructField("nationality", StringType(), True),
    StructField("url", StringType(), True)
])

#### Read with Auto Loader

In [0]:
df_raw = (
    spark.readStream.format('cloudFiles')
        .option('cloudFiles.format', 'csv')
        .option('cloudFiles.schemaLocation', f'{CHECKPOINT_PATH}/schema')
        .option('header', True)
        .option('nullValue', r'\N')
        .schema(drivers_schema)
        .load(RAW_PATH)
)

#### Add metadata columns (source filename, ingestion timestamp, load date)

In [0]:
df_bronze = (
    df_raw
        .withColumn('source_filename', col("_metadata.file_path"))
        .withColumn('ingestion_timestamp', current_timestamp())
        .withColumn('load_date', current_date())
)

#### Load the file as Delta table idempodently

In [0]:
write_query = (
    df_bronze.writeStream.format('delta')
        .outputMode('append')
        .option('checkpointLocation', CHECKPOINT_PATH)
        .trigger(availableNow=True) 
        .toTable(TARGET_TABLE)
)

write_query.awaitTermination()

In [0]:
spark.table(TARGET_TABLE).limit(10).display()